# Data Sanity Checks: BCI Competition IV Dataset 2a

Verify that `load_subject` returns plausible EEG data
before building the preprocessing pipeline.

Three checks:
1. Time-domain trace of one trial — does it look like biology?
2. Power spectrum — does the frequency content look like brain signal?
3. Class-averaged ERPs — does class structure survive averaging?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from neurostream.data.loader import load_subject

# Plot defaults — readable axes, consistent across cells
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


In [ ]:
epochs, labels = load_subject(subject_id=1, session="T")

print(f"Epochs shape: {epochs.shape}")
print(f"Epochs dtype: {epochs.dtype}")
print(f"Epochs max is {epochs.max()}")
print(f"Epochs min is {epochs.min()}")

print(f"Labels shape: {labels.shape}")
print(f"Labels dtype: {labels.dtype}")

unique = np.unique(labels, return_counts=True)
print(f"Unique labels: {unique[0]}")
print(f"Label counts: {unique[1]}")

In [ ]:
# Standard BCI IV 2a montage — 22 EEG channels in 10-20 system positions.
# C3 is over left motor cortex (responds to right-hand imagery).
# We need to find C3's index in the channel ordering.

import mne
raw = mne.io.read_raw_gdf("../data/raw/bci_iv_2a/A01T.gdf", preload=False, verbose="ERROR")
eeg_channel_names = [ch for ch in raw.ch_names if ch.startswith("EEG")]
print("All EEG channels:", eeg_channel_names)

In [ ]:
EEG_C3_IDX = 7
SFREQ = 250  # Hz
print(f"C3 channel name: {eeg_channel_names[EEG_C3_IDX]}")
print(f"Sampling frequency: {SFREQ} Hz")

In [ ]:
trial_idx = 0
class_names = ["left hand", "right hand", "feet", "tongue"]

signal_volts = epochs[trial_idx, EEG_C3_IDX, :]
signal_uv = signal_volts * 1e6  # convert to microvolts for readability
t = np.arange(len(signal_uv)) / SFREQ

fig, ax = plt.subplots()
ax.plot(t, signal_uv, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (μV)")
ax.set_title(
    f"Subject 1, Trial {trial_idx}, C3 — class: {class_names[labels[trial_idx]]}"
)
ax.axvline(1.0, color="red", linestyle="--", alpha=0.5, label="cue onset (t=1s)")
ax.legend()
plt.show()

- Oscillation identified
- Appropriate oscillation range

- A single FFT from one trial shows a noisy power spectrum

- **Welch's method** takes multiple small windows of FFT and averages them to reduce noice and variance of the power spectrum.

In [ ]:
# nperseg=256 gives ~1 Hz frequency resolution at 250 Hz sample rate.
from scipy.signal import welch

freqs, psd_per_trial = welch(
epochs[:, EEG_C3_IDX, :], # all 288 trials at C3 
fs=SFREQ,
nperseg=256,
axis=-1,
)
psd_mean = psd_per_trial.mean (axis=0) # average across trials → (n_freqs,)

fig, ax = plt. subplots (figsize=(10, 5))
ax.semilogy (freqs, psd_mean, linewidth=1.2)
ax.set_xlabel("Frequency (Hz) ")
ax.set_ylabel("Power Spectral Density (V^2/Hz) ")
ax.set_title("C3 power spectrum, averaged over 288 trials (subject 1, session T)")
ax.set_xlim (0, 80)

# Highlight the bands of interest
ax.axvspan(8, 13, alpha=0.15, color="green", label="alpha (8-13 Hz) ")
ax.axvspan (13, 30, alpha=0.15, color="orange", label="beta (13-30 Hz)") 
ax.axvline (50, color="red", alpha=0.5, linestyle="--", label="line noise (50 Hz)")
ax.legend(loc="upper right")
plt.show()

### Key Findings ###
- Alpha bump in the band between 8 - 13 Hz
- A dip around 50 Hz but with a spike on 50 Hz highlights that the data has already been notch-filtered to remove the noise from European electricity frequencies. Notch filter has not fully reduce the spike on 50 Hz

In [ ]:
fig, axes = plt. subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)

for cls in range (4):
    cls_trials = epochs[labels == cls, EEG_C3_IDX, :] * 1e6
    mean = cls_trials.mean (axis=0)
    sem = cls_trials.std(axis=0) / np.sqrt (len(cls_trials))

    ax = axes[cls // 2, cls % 2]
    ax.plot(t, mean, linewidth=1.2)
    ax.fill_between(t, mean - sem, mean + sem, alpha=0.3)
    ax.axvline (1.0, color="red", linestyle="--", alpha=0.4)
    ax.set_title(f"Class {cls}: {class_names[cls]} (n={len(cls_trials)})")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude (pV) ")

fig.suptitle("Class-averaged ERPs at C3", y=1.02)
fig. tight_layout()
plt.show()

## Conclusions
- Time-domain trace shows realistic EEG amplitudes and oscillation
- Power spectrum shows 1/f shape, alpha bump, and 50 Hz line noise spike
- Class-averaged ERPs show distinct waveforms across classes